In [1]:
import os
import re
import ast
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# Load API key from root .env
load_dotenv(dotenv_path="../../../.env")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    load_dotenv()
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found! Please verify your .env file.")

# Set evaluation model (Starting with DeepSeek V4 Flash)
EVAL_MODEL = "google/gemini-3.1-flash-lite"

print(f"Environment ready. Evaluation Model set to: {EVAL_MODEL}")

Environment ready. Evaluation Model set to: google/gemini-3.1-flash-lite


In [2]:
# The FULL dataset generated by GPT-5.6 Luna
INPUT_FILE = "gpt5.6_luna_health_condition_full_dataset.csv"
OUTPUT_FILE = "gpt5.6_luna_full_evaluated_by_gemini_3_1_flash_lite.csv"

print(f"Loading full generated dataset from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

def extract_target_condition(row):
    """Extracts target health condition from available columns."""
    if "target_health_condition" in row and pd.notna(row["target_health_condition"]):
        return str(row["target_health_condition"]).strip()
    
    val = row.get("Health Condition")
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            parsed = ast.literal_eval(val_str)
            if isinstance(parsed, list) and len(parsed) > 0:
                return str(parsed[0]).strip()
        except Exception:
            pass
    return val_str

df["eval_target_condition"] = df.apply(extract_target_condition, axis=1)

# Filter out any rows that failed to generate properly
df = df.dropna(subset=["text", "modified_sentence", "eval_target_condition"]).reset_index(drop=True)
print(f"Loaded {len(df)} rows ready for full evaluation.")

Loading full generated dataset from gpt5.6_luna_health_condition_full_dataset.csv...
Loaded 577 rows ready for full evaluation.


In [3]:
PROMPT_TEMPLATE = """
You are a medical verification assistant.

Determine whether the extracted medical entity is hallucinated with respect to the given text.

Text:
{text}

Extracted Entity:
{target_entity}

If the extracted entity is hallucinated, output:
ANSWER: 1

Otherwise, output:
ANSWER: 0
"""

def evaluate_entity(text, target_entity, model_name=EVAL_MODEL):
    prompt = PROMPT_TEMPLATE.format(
        text=str(text).strip(),
        target_entity=str(target_entity).strip()
    )

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=120
    )
    response.raise_for_status()

    result = response.json()
    if "choices" not in result:
        raise ValueError(f"OpenRouter did not return choices: {result}")

    content = result["choices"][0]["message"]["content"].strip()
    
    match = re.search(r"ANSWER:\s*([01])", content, re.IGNORECASE)
    prediction = int(match.group(1)) if match else None

    return prediction, content

In [4]:
correct_predictions = []
hall_predictions = []

correct_raw = []
hall_raw = []

correct_matches = 0
hall_matches = 0

print(f"Starting FULL evaluation on {len(df)} rows using {EVAL_MODEL}...\n")

for i, row in df.iterrows():
    orig_text = row["text"]
    mod_text = row["modified_sentence"]
    target_condition = row["eval_target_condition"]

    # 1. Evaluate Original Sentence (Expected ANSWER: 0)
    try:
        pred_orig, raw_orig = evaluate_entity(orig_text, target_condition)
    except Exception as e:
        print(f"Row {i+1} [Original] Error: {e}")
        pred_orig, raw_orig = None, str(e)

    correct_predictions.append(pred_orig)
    correct_raw.append(raw_orig)
    if pred_orig == 0:
        correct_matches += 1

    # 2. Evaluate Hallucinated Sentence (Expected ANSWER: 1)
    try:
        pred_hall, raw_hall = evaluate_entity(mod_text, target_condition)
    except Exception as e:
        print(f"Row {i+1} [Hallucinated] Error: {e}")
        pred_hall, raw_hall = None, str(e)

    hall_predictions.append(pred_hall)
    hall_raw.append(raw_hall)
    if pred_hall == 1:
        hall_matches += 1

    print(f"[{i+1}/{len(df)}] Orig Pred: {pred_orig} (Exp: 0) | Hall Pred: {pred_hall} (Exp: 1)")
    
    # AUTO-SAVE LOGIC: Protects against API drops
    if (i + 1) % 50 == 0:
        df_temp = df.loc[:i].copy()
        df_temp["correct_prediction"] = correct_predictions
        df_temp["correct_raw_response"] = correct_raw
        df_temp["hallucinated_prediction"] = hall_predictions
        df_temp["hallucinated_raw_response"] = hall_raw
        df_temp.to_csv("backup_" + OUTPUT_FILE, index=False, encoding="utf-8-sig")
        print(f"   --- Auto-saved backup at row {i+1} ---")

    time.sleep(0.5)

# Final Save and Metric Calculation
df["correct_prediction"] = correct_predictions
df["correct_raw_response"] = correct_raw
df["hallucinated_prediction"] = hall_predictions
df["hallucinated_raw_response"] = hall_raw

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

total_rows = len(df)
correct_accuracy = (correct_matches / total_rows) * 100 if total_rows > 0 else 0
hall_accuracy = (hall_matches / total_rows) * 100 if total_rows > 0 else 0

print("\n==============================")
print(f"Evaluator Model                : {EVAL_MODEL}")
print(f"Correct Sentence Accuracy      : {correct_accuracy:.2f}%")
print(f"Hallucinated Sentence Accuracy : {hall_accuracy:.2f}%")
print(f"Saved detailed results to      : {OUTPUT_FILE}")
print("==============================")

Starting FULL evaluation on 577 rows using google/gemini-3.1-flash-lite...

[1/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[2/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[3/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[4/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[5/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[6/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[7/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[8/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[9/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[10/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[11/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[12/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[13/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[14/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[15/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[16/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[17/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1